## PCA example

#### Stiefel Manifold: $St(d,k) = \{U \in \mathbb{R}^{d\times k} : U^TU = I_k\}$

#### $f(U) = \langle XX^TU,U \rangle = tr(U^TXX^TU)$

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA

In [42]:
def f(U, S):
  return -np.sum(U.T @ S @ U * np.eye(U.shape[1]))

def grad_f(U, S):
  G = -2 * S @ U
  UtG = U.T @ G
  symUtG = (UtG + UtG.T) / 2
  return G - U @ symUtG

def qr_retraction(Y):
  Q, R = np.linalg.qr(Y)
  d = np.sign(np.diag(R))
  d[d == 0]= 1
  Q = Q @ np.diag(d) 
  return Q

In [50]:
def pca_stiefel(X, k, max_iters=5000, tau = .5, r = 10e-3, alpha = .2):
  n = X.shape[0]
  Xc = X - X.mean(axis = 1, keepdims=True)
  S = Xc @ Xc.T
  #S = X @ X.T
  U = np.linalg.qr(np.random.randn(n, k))[0]

  for _ in range(max_iters):
    G = grad_f(U, S)
    if np.sum(G*G) < 1e-10:
      break
    
    f_curr = f(U,S)
    alpha_k = alpha

    while True:
      Y = U - alpha_k * G
      U_new = qr_retraction(Y)
      f_new = f(U_new, S)

      if f_new <= f_curr - r * alpha_k * np.sum(G*G):
        break

      alpha_k *= tau
      if alpha_k < 1e-10:
        U_new = U
        break

    U = U_new

  return U

In [ ]:
data = load_iris()
X = data.data.T  
k = 2

U_riem = pca_stiefel(X, k)

pca = PCA(n_components=k)
pca.fit(X.T)
U_sklearn = pca.components_.T   

P_riem = U_riem @ U_riem.T
P_sklearn = U_sklearn @ U_sklearn.T
print("Subspace error:", np.linalg.norm(P_riem - P_sklearn, 'fro'))


Subspace error: 2.9819448715673775e-09
